# 03 · Cashflows, Schedules, and Builders

The valuations cashflow builder mirrors the Rust engine's composable schedule machinery. It lets you stitch together principal profiles, fixed/floating coupons, amortization programs, and PIK toggles before promoting the schedule into an instrument (e.g., `Bond.from_cashflows`).

### Learning Objectives
- Use `CashflowBuilder` with schedule presets and coupon specs to create deterministic flows
- Mix fixed, floating, and split (cash/PIK) coupons with optional amortization
- Apply schedule programs such as step-up coupons and payment split transitions
- Feed a custom schedule directly into the valuation stack for pricing and risk

In [ ]:
from datetime import date

from finstack import Money
from finstack.core.currency import USD, EUR
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve, ForwardCurve
from finstack.valuations.cashflow import (
    CashflowBuilder,
    ScheduleParams,
    FixedCouponSpec,
    FloatCouponParams,
    FloatingCouponSpec,
    CouponType,
    AmortizationSpec,
)
from finstack.valuations.instruments import Bond
from finstack.valuations.pricer import create_standard_registry

as_of = date(2025, 1, 2)

def preview_schedule(schedule, rows=8):
    flows = schedule.flows()
    for idx, flow in enumerate(flows[:rows]):
        dt, amt, kind, accrual, reset = flow.to_tuple()
        reset_str = reset.isoformat() if reset else "-"
        print(
            f"{idx:02d} {dt} {str(kind):<12} {amt.amount:>12,.2f} {amt.currency} "
            f"accr={accrual:>7.5f} reset={reset_str}"
        )
    if len(flows) > rows:
        print(f"... {len(flows) - rows} additional cashflows")


## 1. Fixed-Rate Schedule Basics
Start with a quarterly 5% coupon bond. The builder takes the principal profile plus a `FixedCouponSpec`. Schedule presets (`ScheduleParams.quarterly_act360()`) encapsulate frequency/day-count/BDC settings.

In [ ]:
fixed_builder = CashflowBuilder.new()
fixed_builder.principal(
    amount=1_000_000,
    currency=USD,
    issue=as_of,
    maturity=date(2027, 1, 2),
)
fixed_spec = FixedCouponSpec.new(
    rate=0.05,
    schedule=ScheduleParams.quarterly_act360(),
    coupon_type=CouponType.CASH,
)
fixed_builder.fixed_cf(fixed_spec)
fixed_schedule = fixed_builder.build_with_curves(None)
preview_schedule(fixed_schedule)


## 2. Floating-Rate Coupons With Curves
`build_with_curves` pulls forward rates from the supplied `MarketContext`, applying margins and reset lags. Here we create a SOFR 3M FRN with +150 bp spread.

In [ ]:
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (1.0, 0.995), (3.0, 0.980), (5.0, 0.950)],
    )
)
market.insert_forward(
    ForwardCurve(
        "USD-SOFR-3M",
        0.25,
        [(0.0, 0.045), (1.0, 0.0465), (3.0, 0.0485), (5.0, 0.0500)],
        base_date=as_of,
    )
)

float_builder = CashflowBuilder.new()
float_builder.principal(
    amount=5_000_000,
    currency=USD,
    issue=as_of,
    maturity=date(2028, 1, 2),
)
float_params = FloatCouponParams.new(
    index_id="USD-SOFR-3M",
    margin_bp=150.0,
    gearing=1.0,
    reset_lag_days=2,
)
float_spec = FloatingCouponSpec.new(
    params=float_params,
    schedule=ScheduleParams.quarterly_act360(),
    coupon_type=CouponType.CASH,
)
float_builder.floating_cf(float_spec)
floating_schedule = float_builder.build_with_curves(market)
preview_schedule(floating_schedule)


## 3. Cash vs PIK Splits
Coupon types support `%` weights between cash and payment-in-kind. This example allocates 70% of the 8% coupon in cash and 30% capitalized back into principal.

In [ ]:
pik_builder = CashflowBuilder.new()
pik_builder.principal(
    amount=2_000_000,
    currency=EUR,
    issue=as_of,
    maturity=date(2030, 1, 2),
)
pik_spec = FixedCouponSpec.new(
    rate=0.08,
    schedule=ScheduleParams.semiannual_30360(),
    coupon_type=CouponType.split(0.70, 0.30),
)
pik_builder.fixed_cf(pik_spec)
pik_schedule = pik_builder.build_with_curves(None)
preview_schedule(pik_schedule, rows=10)


## 4. Amortization Profiles
`AmortizationSpec` covers linear ramps, step targets, or custom principal grids. Below we repay to 40% of notional over the final year while continuing quarterly coupons.

In [ ]:
amort_builder = CashflowBuilder.new()
amort_builder.principal(
    amount=3_000_000,
    currency=USD,
    issue=as_of,
    maturity=date(2029, 1, 2),
)
amort_builder.amortization(AmortizationSpec.linear_to(Money(1_200_000, USD)))
amort_spec = FixedCouponSpec.new(
    rate=0.045,
    schedule=ScheduleParams.quarterly_act360(),
    coupon_type=CouponType.CASH,
)
amort_builder.fixed_cf(amort_spec)
amort_schedule = amort_builder.build_with_curves(None)
preview_schedule(amort_schedule, rows=10)


## 5. Step Programs & Payment Split Transitions
Schedules can change coupon levels or splits on arbitrary dates. The following builder steps coupons from 4% → 5.5% → 6.5% and flips from PIK-only to cash collections after year 1.

In [ ]:
program_builder = CashflowBuilder.new()
program_builder.principal(
    amount=1_500_000,
    currency=USD,
    issue=as_of,
    maturity=date(2028, 1, 2),
)
# Use fixed_stepup to define step-up coupon schedule
# The steps define period boundaries: [(end_date, rate), ...]
# Implicit first period from issue to first step date uses initial rate
program_builder.fixed_stepup(
    steps=[
        (date(2026, 1, 2), 0.04),  # 4% from issue to 2026-01-02
        (date(2027, 7, 2), 0.055), # 5.5% from 2026-01-02 to 2027-07-02  
        (date(2028, 1, 2), 0.065), # 6.5% from 2027-07-02 to maturity
    ],
    schedule=ScheduleParams.quarterly_act360(),
    default_split=CouponType.PIK,  # Start with PIK
)
# Transition payment splits from PIK to CASH
program_builder.payment_split_program(
    [
        (date(2026, 1, 2), CouponType.CASH),
    ]
)
program_schedule = program_builder.build_with_curves(None)
preview_schedule(program_schedule, rows=12)


## 6. Promote a Schedule Into an Instrument
Schedules can be fed straight into the valuation stack. Here we wrap the amortizing schedule as a `Bond` and request clean price / DV01 metrics using the same market context from the floating example.

In [ ]:
registry = create_standard_registry()
custom_bond = Bond.from_cashflows(
    instrument_id="CUSTOM-CF",
    schedule=amort_schedule,
    discount_curve="USD-OIS",
    quoted_clean=99.25,
)
bond_result = registry.price_with_metrics(
    custom_bond,
    "discounting",
    market,
    ["clean_price", "dv01", "duration_mod"],
    as_of=as_of,
)
print("PV:", bond_result.value)
print("Clean price:", bond_result.measures.get("clean_price"))
print("DV01:", bond_result.measures.get("dv01"))


## Summary
- The builder wires principal, coupon specs, amortization, and payment policies in a readable, chain-friendly API.
- Floating coupons pull index projections directly from `MarketContext` via `build_with_curves`.
- Step programs and payment splits handle practical structures (cash-to-PIK toggles, coupon ramps).
- Any finished schedule can flow into `Bond.from_cashflows`, a structured product waterfall, or other valuation entry points.